# CodeTuneEfficiency on a free GPU

Runs the full benchmark on **Google Colab (free T4)** or **Kaggle (free weekly GPU hours)**.
Nothing here costs money and no paid tier is required.

| | Colab free | Kaggle free |
|---|---|---|
| GPU | T4, 16 GB | T4 x2 or P100, 16 GB |
| Session limit | ~12 h | 12 h, 30 GPU-h/week |
| Enable it | Runtime → Change runtime type → T4 GPU | Settings → Accelerator → GPU |

With 16 GB instead of the 6 GB this was developed on you can raise `batch_size` to 32
and drop `grad_accum` to 1, which is roughly 3x faster.

**Download `results/` before the session ends — Colab and Kaggle both wipe the disk on disconnect.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Colab and Kaggle already ship a CUDA build of torch, so only the rest is installed.
!git clone https://github.com/Ssavan99/CodeTuneEfficiency.git
%cd CodeTuneEfficiency
!pip install -q transformers==4.44.2 datasets==2.21.0 peft==0.12.0 accelerate==0.34.2 pyyaml==6.0.2

In [ ]:
# Devign comes from the public CodeXGLUE copy on the Hub (free, no account).
# Clone detection data ships with the repo.
!python -m codetune prepare

In [ ]:
# Sanity check first - a couple of minutes, proves the pipeline works.
!python -m codetune run --config configs/smoke.yaml

In [ ]:
# A 16 GB card fits a much larger batch than the 6 GB card this was written on.
import yaml

for name in ("configs/defect.yaml", "configs/clone.yaml"):
    cfg = yaml.safe_load(open(name))
    cfg["batch_size"], cfg["grad_accum"] = 32, 1
    yaml.safe_dump(cfg, open(name, "w"), sort_keys=False)
    print(name, "-> batch_size=32, grad_accum=1")

In [ ]:
# 4 methods x 3 seeds. `grid` skips runs that already have a result file, so
# re-running after a disconnect resumes instead of starting over.
!python -m codetune grid --config configs/defect.yaml

In [ ]:
!python -m codetune grid --config configs/clone.yaml

In [ ]:
!python -m codetune aggregate
!python -m codetune plot

In [ ]:
# Save the results off the ephemeral disk before the session ends.
!zip -qr results.zip results && echo "wrote results.zip"
try:
    from google.colab import files

    files.download("results.zip")
except ImportError:
    print("On Kaggle: results.zip is in the working directory - use the Output panel to download it.")